# Fast Classifier v2 — 7-category taxonomy

Same inputs, same 7 labels, same kind of `.joblib` bundle as v1. The changes:

| Area | v1 | v2 |
|---|---|---|
| **Labels** | Empty or unrecognized labels were quietly put into *Non-Music/Other* | Empty labels are dropped, unrecognized ones are printed (you choose what happens to them), and matching ignores case and dash style |
| **Duplicates** | Not handled, so the same video could sit in train *and* val | Duplicates are removed before the split, and conflicting labels are exported for review |
| **Numbers** | Raw counts ×MaxAbs, so a 3-min video's duration ≈ 0.00 (the signal was lost). `"1,234"` → NaN → 0 | log1p + standardize, plus duration buckets (Shorts / 3-5 min / 1h+ mixes…). Commas are handled and hidden likes (-1) become missing values |
| **Text** | Every field joined into one blob | Separate TF-IDF per field: title words (1-3) + title chars, description, each tag field, and creator-name chars (`VEVO`, `- Topic`, `Lyrics`…) |
| **Classifier** | SGD, max 50 epochs, 10-candidate search | SGD / LinearSVC / LogReg compared on the same CV folds. The winner is calibrated for probabilities |
| **Evaluation** | One random split | Also compares against v1 on the same split, tests on creators the model hasn't seen, exports errors, and runs an out-of-fold label-noise review |

**Run:** Runtime → Run all, then upload the CSV (or set `CSV_PATH`). You'll get `fast_linear_tfidf_v2.joblib`, `val_errors.csv`, `label_review.csv`, and `label_conflicts.csv` (that last one only if conflicts are found).

In [ ]:
import io, re, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import (GroupShuffleSplit, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, MaxAbsScaler, OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# -------- Run options --------
CSV_PATH = None             # e.g. "/content/drive/MyDrive/train.csv" to skip the upload prompt
UNMAPPED_LABELS = "other"   # labels not in the taxonomy map: "other" → catch-all class (v1 behaviour), "drop" → exclude
DEDUPE = True               # remove duplicate videos before splitting (dupes across train/val inflate scores)
SELECT_BY = "f1_macro"      # model-selection metric: "f1_macro" or "accuracy"
TRY_LOGREG = False          # also try LogisticRegression (often ≈ LinearSVC, but several × slower)
RUN_CREATOR_HOLDOUT = True  # extra eval on creators never seen in training
RUN_LABEL_REVIEW = True     # out-of-fold pass over all rows → label_review.csv of likely mislabels
REFIT_ON_ALL_DATA = True    # saved model is refit on train+val after evaluation
RANDOM_STATE = 42
ARTIFACT = "fast_linear_tfidf_v2.joblib"

In [ ]:
# -------- Load CSV --------
def robust_read_csv(name: str, content: bytes) -> pd.DataFrame:
    bio = io.BytesIO(content)
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        bio.seek(0)
        try:
            return pd.read_csv(bio, encoding=enc, sep=None, engine="python",
                               dtype=str, keep_default_na=False)
        except Exception:
            pass
    for sep in ("\t", ";", "|", ","):
        bio.seek(0)
        try:
            return pd.read_csv(bio, encoding="utf-8", sep=sep,
                               dtype=str, keep_default_na=False)
        except Exception:
            pass
    raise RuntimeError(f"Could not parse {name} as CSV")

if CSV_PATH:
    with open(CSV_PATH, "rb") as fh:
        file_name, raw_bytes = CSV_PATH, fh.read()
else:
    from google.colab import files
    print("Upload your labeled training CSV …")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    file_name, raw_bytes = next(iter(uploaded.items()))

df = robust_read_csv(file_name, raw_bytes)
df.columns = pd.Index(df.columns).astype(str).str.replace("﻿", "", regex=False).str.strip()
df = df.loc[:, ~df.columns.str.match(r"^Unnamed:\s*\d+$", na=False)].copy()
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns from {file_name}")

cols_lower = pd.Index(df.columns.str.lower().str.strip())
def has_col(name): return name.lower() in cols_lower
def get_col(name): return df.columns[cols_lower.get_loc(name.lower())] if has_col(name) else None

In [ ]:
# -------- Labels → 7-category taxonomy (with an audit of what doesn't map) --------
label_src = next((get_col(c) for c in ["label", "category"] if has_col(c)), None)
if label_src is None:
    raise ValueError("No ground-truth column found. Include a 'label' column (or 'Label'/'category').")
df = df.rename(columns={label_src: "label"})

def normalize_dash(s: str) -> str:
    if not isinstance(s, str): s = "" if pd.isna(s) else str(s)
    s = s.replace("‚Äî", "—").replace("â€”", "—").replace("--", "—")
    return re.sub(r"\s+", " ", s).strip()

ORDERED_LABELS = [
    "Clipped/Short Music — Official Music Video",
    "Clipped/Short Music — Remix",
    "Clipped/Short Music — Lyric videos",
    "Performances",
    "Critiques / Breakdown / Reaction/Interviews",
    "Ambient / Mixes",
    "Non-Music/Other Music Related / Fan Created",
]
CATCH_ALL = ORDERED_LABELS[-1]

map_old_to_new = {
    "Clipped/Short Music — Official Music Video": "Clipped/Short Music — Official Music Video",
    "Ambient / Mixes": "Ambient / Mixes",
    "Clipped/Short Music — Remix": "Clipped/Short Music — Remix",
    "Remix": "Clipped/Short Music — Remix",
    "Clipped/Short Music — Lyric Video": "Clipped/Short Music — Lyric videos",
    "Clipped/Short Music — Lyric Videos": "Clipped/Short Music — Lyric videos",
    "Lyric Video": "Clipped/Short Music — Lyric videos",
    "Lyric Videos": "Clipped/Short Music — Lyric videos",
    "Performances": "Performances",
    "Clipped/Short Music — Artist-released clipped performance videos": "Performances",
    "Full-length Performances": "Performances",
    "Full-length Performances — Concert Perfomance Footage": "Performances",
    "Full-length Performances — Performance-based Music Competition": "Performances",
    "Full-length Performances — Talk show / variety": "Performances",
    "Artist interviews": "Critiques / Breakdown / Reaction/Interviews",
    "Interviews / Critiques — Artist interviews": "Critiques / Breakdown / Reaction/Interviews",
    "Interviews / Critiques — Album/song reaction or review": "Critiques / Breakdown / Reaction/Interviews",
    "Interviews / Critiques — Breakdown / Explainer": "Critiques / Breakdown / Reaction/Interviews",
    "Critiques / Breakdown / Reaction": "Critiques / Breakdown / Reaction/Interviews",
    "Non-Music": "Non-Music/Other Music Related / Fan Created",
    "Other Music Related": "Non-Music/Other Music Related / Fan Created",
    "Clipped/Short Music — Fan-Created / fan-edit videos": "Non-Music/Other Music Related / Fan Created",
    "Fictional Movies/Shows": "Non-Music/Other Music Related / Fan Created",
    "Fictional Movies/Shows — Biopic": "Non-Music/Other Music Related / Fan Created",
    "Fictional Movies/Shows — Musical": "Non-Music/Other Music Related / Fan Created",
    "Fictional Movies/Shows — Other Fictional": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Biographical Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Genre-Focused Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Period-Focused Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Fan culture Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Behind-the-scenes Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Reality TV Documentary": "Non-Music/Other Music Related / Fan Created",
    "Documentaries — Other Music Documentary": "Non-Music/Other Music Related / Fan Created",
    "Uncategorized": "Non-Music/Other Music Related / Fan Created",
}

def canon(s: str) -> str:
    s = normalize_dash(s).lower()
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

# Both lookups are case/dash/spacing-insensitive (v1 only did that for the final names)
canon_map = {canon(k): v for k, v in map_old_to_new.items()}
canon_map.update({canon(v): v for v in ORDERED_LABELS})

raw_label = df["label"].map(normalize_dash)
mapped = raw_label.map(lambda s: canon_map.get(canon(s)))

empty = raw_label.eq("")
if empty.any():
    print(f"Dropping {empty.sum():,} rows with an EMPTY label (v1 trained on these as '{CATCH_ALL}').")

unmapped = mapped.isna() & ~empty
if unmapped.any():
    print(f"⚠️ {unmapped.sum():,} rows have labels not in the taxonomy map → policy UNMAPPED_LABELS='{UNMAPPED_LABELS}'.")
    print("   Add real ones to map_old_to_new; v1 silently trained on all of these as the catch-all class.")
    display(raw_label[unmapped].value_counts().head(25).rename_axis("raw label").to_frame("rows"))
    if UNMAPPED_LABELS == "other":
        mapped = mapped.mask(unmapped, CATCH_ALL)

df = df.loc[mapped.notna()].copy()
df["label"] = mapped[mapped.notna()]

In [ ]:
# -------- De-duplicate videos (before any split) --------
ID_CANDIDATES = ["video_id", "videoid", "youtube_id", "id", "video_url", "url", "link"]
id_col = next((get_col(c) for c in ID_CANDIDATES if has_col(c)), None)

if DEDUPE:
    row_key = pd.Series("row" + df.index.astype(str), index=df.index)   # unique fallback key
    if id_col:
        key = df[id_col].astype(str).str.strip()
        key_desc = f"'{id_col}'"
    else:
        title_norm = df[get_col("Title")].astype(str).str.lower().str.strip()
        key = (title_norm + "\x1f" + df[get_col("creator_name")].astype(str).str.lower().str.strip()
               + "\x1f" + df[get_col("duration_sec")].astype(str).str.strip())
        key_desc = "title + creator + duration"
        key = key.where(title_norm.ne(""), row_key)
    key = key.where(key.ne(""), row_key)

    n_labels = df.groupby(key)["label"].transform("nunique")
    conflicted = n_labels > 1
    if conflicted.any():
        cols = [c for c in [id_col, get_col("Title"), get_col("creator_name")] if c] + ["label"]
        df.loc[conflicted, cols].assign(_key=key[conflicted]).sort_values("_key").to_csv("label_conflicts.csv", index=False)
        majority = df.loc[conflicted].groupby(key[conflicted])["label"].agg(lambda s: s.value_counts().index[0])
        df["label"] = key.map(majority).fillna(df["label"])

    before = len(df)
    keep = ~key.duplicated()
    df = df.loc[keep].copy()
    print(f"Dedupe on {key_desc}: {before:,} → {len(df):,} rows ({before - len(df):,} duplicates removed).")
    if conflicted.any():
        print(f"   {key[conflicted].nunique():,} videos had CONFLICTING labels → majority vote; see label_conflicts.csv")

print("\nClass counts:")
display(df["label"].value_counts().reindex(ORDERED_LABELS).fillna(0).astype(int).to_frame("rows"))

In [ ]:
# -------- Features, numeric parsing, split --------
REQ_TEXT = ["Title", "Description", "Topics_csv", "Categories_csv", "Keywords_csv"]
REQ_CAT  = ["creator_name", "youtube_category"]
REQ_NUM  = ["Views", "Likes", "Comments", "engagements_total", "duration_sec"]

missing = [c for c in (REQ_TEXT + REQ_CAT + REQ_NUM) if not has_col(c)]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

FEATS_TEXT = [get_col(c) for c in REQ_TEXT]
FEATS_CAT  = [get_col(c) for c in REQ_CAT]
FEATS_NUM  = [get_col(c) for c in REQ_NUM]

for c in FEATS_TEXT + FEATS_CAT:
    df[c] = df[c].fillna("").astype(str)
for c in FEATS_NUM:
    # "1,234,567" → 1234567 (v1 turned these into NaN → 0); negatives such as -1 for hidden likes → missing
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce")
    df.loc[df[c] < 0, c] = np.nan

print("Numeric columns — % missing/unparseable (high values here mean lost signal):")
display((df[FEATS_NUM].isna().mean() * 100).round(1).to_frame("% missing"))

label_to_idx = {l: i for i, l in enumerate(ORDERED_LABELS)}
X = df[FEATS_TEXT + FEATS_CAT + FEATS_NUM]
y = df["label"].map(label_to_idx).to_numpy()

try:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
except ValueError:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE)
    print("⚠️ Stratified split unavailable (rare classes). Proceeding without stratify.")
print(f"Train rows: {len(X_train):,} | Val rows: {len(X_val):,}")

In [ ]:
# -------- Sparse preprocessor: one block per field --------
TITLE, DESC = get_col("Title"), get_col("Description")
TAG_COLS = [get_col(c) for c in ["Topics_csv", "Categories_csv", "Keywords_csv"]]
CREATOR, YT_CATEGORY = get_col("creator_name"), get_col("youtube_category")
COUNT_COLS = [get_col(c) for c in ["Views", "Likes", "Comments", "engagements_total"]]
DURATION = get_col("duration_sec")

# Duration buckets (seconds): missing | ≤1m (Shorts) | ≤2m | ≤3m | ≤4m | ≤5m | ≤7m | ≤10m | ≤15m | ≤30m | ≤1h | ≤2h | >2h
DURATION_EDGES = [0, 61, 121, 181, 241, 301, 421, 601, 901, 1801, 3601, 7201]

def tfidf(**kw):
    return TfidfVectorizer(sublinear_tf=True, dtype=np.float32, **kw)

text_blocks = [
    ("title_words",   tfidf(ngram_range=(1, 3), min_df=2, max_features=150_000), TITLE),
    ("title_chars",   tfidf(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=100_000), TITLE),
    ("desc_words",    tfidf(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=150_000), DESC),
    ("creator_chars", tfidf(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=30_000), CREATOR),
] + [(f"tags_{c}", tfidf(ngram_range=(1, 2), min_df=2, max_features=50_000), c) for c in TAG_COLS]

# Skip near-empty text fields (TfidfVectorizer errors on an empty vocabulary)
min_rows = max(100, int(0.01 * len(df)))
skipped = [name for name, _, col in text_blocks if (df[col].str.strip() != "").sum() < min_rows]
text_blocks = [b for b in text_blocks if b[0] not in skipped]
if skipped:
    print(f"Skipping near-empty text fields: {skipped}")

preproc = ColumnTransformer(
    transformers=text_blocks + [
        ("creator", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2), [CREATOR]),
        ("yt_category", OneHotEncoder(handle_unknown="ignore"), [YT_CATEGORY]),
        ("log_numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("clip", FunctionTransformer(np.clip, kw_args={"a_min": 0, "a_max": None}, feature_names_out="one-to-one")),
            ("log1p", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
            ("scale", StandardScaler()),
        ]), COUNT_COLS + [DURATION]),
        ("duration_bucket", Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value=-1)),
            ("bucket", FunctionTransformer(np.digitize, kw_args={"bins": DURATION_EDGES}, feature_names_out="one-to-one")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), [DURATION]),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

t0 = time.time()
Xtr = preproc.fit_transform(X_train, y_train)
Xva = preproc.transform(X_val)
print(f"Feature matrix: {Xtr.shape[1]:,} columns ({time.time() - t0:.0f}s)")
for name, trans, _ in preproc.transformers_:
    if hasattr(trans, "vocabulary_"):
        print(f"  {name:<28} {len(trans.vocabulary_):>8,} terms")

In [ ]:
# -------- Compare classifiers on identical CV folds (features cached → fast) --------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

candidates = {
    "SGD log-loss (v1 settings)": SGDClassifier(
        loss="log_loss", alpha=1.35e-5, class_weight="balanced", max_iter=50,
        early_stopping=True, n_iter_no_change=3, tol=1e-3, random_state=RANDOM_STATE),
}
for C in (0.1, 0.3, 1.0):
    for cw in ("balanced", None):
        candidates[f"LinearSVC C={C} class_weight={cw}"] = LinearSVC(C=C, class_weight=cw, max_iter=5000)
if TRY_LOGREG:
    for C in (10.0, 30.0):
        candidates[f"LogReg C={C} class_weight=balanced"] = LogisticRegression(
            C=C, class_weight="balanced", max_iter=1000)

rows = []
for name, est in candidates.items():
    t0 = time.time()
    res = cross_validate(est, Xtr, y_train, cv=cv, scoring=["f1_macro", "accuracy"], n_jobs=-1)
    rows.append({"model": name,
                 "cv_f1_macro": res["test_f1_macro"].mean(),
                 "cv_accuracy": res["test_accuracy"].mean(),
                 "sec": time.time() - t0})
    print(f"{name:<40} f1_macro={rows[-1]['cv_f1_macro']:.4f}  acc={rows[-1]['cv_accuracy']:.4f}  ({rows[-1]['sec']:.0f}s)")

cv_table = pd.DataFrame(rows).sort_values(f"cv_{SELECT_BY}", ascending=False).reset_index(drop=True)
display(cv_table.round(4))

best_name = cv_table.loc[0, "model"]
best_est = candidates[best_name]
# LinearSVC has no predict_proba → wrap it in a calibrator so the bundle still exposes probabilities
final_clf = (CalibratedClassifierCV(clone(best_est), method="sigmoid", cv=5)
             if isinstance(best_est, LinearSVC) else clone(best_est))
print(f"\nSelected by CV {SELECT_BY}: {best_name}"
      + ("  (+ sigmoid calibration for probabilities)" if isinstance(best_est, LinearSVC) else ""))

In [ ]:
# -------- Held-out validation --------
final_clf.fit(Xtr, y_train)
best_pipe = Pipeline([("preproc", preproc), ("clf", final_clf)])   # both steps already fitted on X_train

val_proba = final_clf.predict_proba(Xva)
val_pred = final_clf.classes_[val_proba.argmax(axis=1)]   # map columns → class ids (safe if a class is absent)

print("=== Classification Report (7 classes) ===")
print(classification_report(y_val, val_pred, labels=np.arange(len(ORDERED_LABELS)),
                            target_names=ORDERED_LABELS, zero_division=0, digits=3))

print("=== Confusion Matrix (rows=true, cols=pred) ===")
short = ["OMV", "Remix", "Lyric", "Perf", "Critique/Interview", "Ambient/Mix", "Other"]
cm = confusion_matrix(y_val, val_pred, labels=np.arange(len(ORDERED_LABELS)))
display(pd.DataFrame(cm, index=short, columns=short))

val_acc = accuracy_score(y_val, val_pred)
val_f1 = f1_score(y_val, val_pred, average="macro")
print(f"\nTop-1 accuracy: {val_acc:.3f} | macro-F1: {val_f1:.3f}")
top2 = (np.argsort(-val_proba, axis=1)[:, :2] == np.searchsorted(final_clf.classes_, y_val)[:, None]).any(axis=1).mean()
print(f"Top-2 accuracy: {top2:.3f}")

In [ ]:
# -------- Apples-to-apples: v1's pipeline (its best params) on THIS split --------
# The table v1 printed came from a different (non-deduped) split, so compare against this.
TEXT_JOIN = "__all_text__"
def with_join(frame):
    frame = frame.copy()
    frame[TEXT_JOIN] = frame[FEATS_TEXT].agg(" | ".join, axis=1)
    return frame

v1_pipe = Pipeline([
    ("preproc", ColumnTransformer([
        ("text_words", TfidfVectorizer(max_features=120_000, ngram_range=(1, 2), min_df=2, sublinear_tf=True), TEXT_JOIN),
        ("text_chars", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=8, max_features=40_000), TEXT_JOIN),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), FEATS_CAT),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
                          ("scale", MaxAbsScaler())]), FEATS_NUM),
    ], sparse_threshold=1.0)),
    ("clf", SGDClassifier(loss="log_loss", class_weight="balanced", alpha=1.35e-5, penalty="l2", max_iter=50,
                          early_stopping=True, n_iter_no_change=3, tol=1e-3, random_state=RANDOM_STATE)),
])
v1_pipe.fit(with_join(X_train), y_train)
v1_pred = v1_pipe.predict(with_join(X_val))

per_class = pd.DataFrame({
    "v1 F1": f1_score(y_val, v1_pred, average=None, labels=np.arange(len(ORDERED_LABELS)), zero_division=0),
    "v2 F1": f1_score(y_val, val_pred, average=None, labels=np.arange(len(ORDERED_LABELS)), zero_division=0),
}, index=ORDERED_LABELS)
per_class.loc["— accuracy"] = [accuracy_score(y_val, v1_pred), val_acc]
per_class.loc["— macro-F1"] = [f1_score(y_val, v1_pred, average="macro"), val_f1]
per_class["Δ"] = per_class["v2 F1"] - per_class["v1 F1"]
display(per_class.round(3))

In [ ]:
# -------- Diagnostics --------
# (a) Unseen creators. creator_name is a strong feature. A random split puts the same creator in train
#     and val, so if you'll classify channels the model hasn't seen, THIS is the number to trust.
if RUN_CREATOR_HOLDOUT:
    groups = X[CREATOR].str.lower().str.strip()
    tr_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE).split(X, y, groups))
    pipe_g = Pipeline([("preproc", clone(preproc)), ("clf", clone(final_clf))]).fit(X.iloc[tr_idx], y[tr_idx])
    pred_g = pipe_g.predict(X.iloc[te_idx])
    print(f"Unseen-creator holdout: accuracy={accuracy_score(y[te_idx], pred_g):.3f} "
          f"macro-F1={f1_score(y[te_idx], pred_g, average='macro'):.3f}  "
          f"(random split: {val_acc:.3f} / {val_f1:.3f})")

# (b) Validation errors, most-confident first (a confident wrong prediction is often a labeling mistake)
val_out = X_val[[TITLE, CREATOR, DURATION]].copy()
val_out["true"] = [ORDERED_LABELS[i] for i in y_val]
val_out["pred"] = [ORDERED_LABELS[i] for i in val_pred]
val_out["confidence"] = val_proba.max(axis=1).round(3)
val_errors = val_out[val_out["true"] != val_out["pred"]].sort_values("confidence", ascending=False)
val_errors.to_csv("val_errors.csv")
print(f"\n{len(val_errors):,} validation errors → val_errors.csv. Most confident:")
display(val_errors.head(15))

In [ ]:
# (c) Label review. Every row gets an out-of-fold prediction; rows where the model gives the assigned
#     label < 10% probability are the best candidates to re-check by hand. Fixing them helps the model
#     AND makes the scores trustworthy.
if RUN_LABEL_REVIEW:
    t0 = time.time()
    oof = cross_val_predict(Pipeline([("preproc", clone(preproc)), ("clf", clone(final_clf))]),
                            X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                            method="predict_proba", n_jobs=-1)
    p_given = oof[np.arange(len(y)), y]
    suggested = oof.argmax(axis=1)
    review = df.loc[:, [c for c in [id_col, TITLE, CREATOR, DURATION] if c]].copy()
    review["given_label"] = df["label"].values
    review["suggested_label"] = [ORDERED_LABELS[i] for i in suggested]
    review["p_given"] = p_given.round(3)
    review["p_suggested"] = oof.max(axis=1).round(3)
    review = review[(p_given < 0.10) & (suggested != y)].sort_values("p_given")
    review.to_csv("label_review.csv")
    print(f"{len(review):,} rows ({len(review) / len(df):.1%}) flagged → label_review.csv  ({time.time() - t0:.0f}s)")
    display(review.groupby(["given_label", "suggested_label"]).size().sort_values(ascending=False).head(10).to_frame("rows"))
    display(review.head(15))

In [ ]:
# -------- Final fit + save (.joblib, same bundle keys as v1) --------
if REFIT_ON_ALL_DATA:
    t0 = time.time()
    final_pipe = Pipeline([("preproc", clone(preproc)), ("clf", clone(final_clf))]).fit(X, y)
    print(f"Refit on all {len(X):,} rows ({time.time() - t0:.0f}s)")
else:
    final_pipe = best_pipe

payload = {
    "model": final_pipe,                   # full pipeline: raw columns in → predict_proba out
    "labels": ORDERED_LABELS,              # model.classes_ are indices into this list
    "features": {
        "text": FEATS_TEXT,
        "cat": FEATS_CAT,
        "num": FEATS_NUM,
        "text_join": "__all_text__",       # not used by v2 (extra columns are ignored); kept so v1 inference code runs
    },
    "best_params": {"model": best_name, **{k: v for k, v in best_est.get_params().items()
                                           if k in ("C", "alpha", "class_weight")}},
    "metrics": {"val_accuracy": val_acc, "val_f1_macro": val_f1, "cv": cv_table.round(4).to_dict("records")},
    "version": {
        "sklearn": __import__("sklearn").__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}
dump(payload, ARTIFACT)
print(f"Saved model bundle → {ARTIFACT}")
try:
    from google.colab import files
    for f in [ARTIFACT, "val_errors.csv", "label_review.csv", "label_conflicts.csv"]:
        try:
            files.download(f)
        except Exception:
            pass
except ImportError:
    pass

### Using the saved model

```python
from joblib import load
import numpy as np, pandas as pd

bundle = load("fast_linear_tfidf_v2.joblib")
model, labels, feats = bundle["model"], bundle["labels"], bundle["features"]

X_new = new_df[feats["text"] + feats["cat"] + feats["num"]].copy()
for c in feats["text"] + feats["cat"]:
    X_new[c] = X_new[c].fillna("").astype(str)
for c in feats["num"]:
    X_new[c] = pd.to_numeric(X_new[c].astype(str).str.replace(r"[,\s]", "", regex=True), errors="coerce")
    X_new.loc[X_new[c] < 0, c] = np.nan

proba = model.predict_proba(X_new)
new_df["pred_label"] = [labels[i] for i in model.classes_[proba.argmax(axis=1)]]
new_df["confidence"] = proba.max(axis=1)
```